# ProteinMPNN — Rediseño de scaffold de RuBisCO

**ProteinMPNN** rediseña la secuencia de aminoacidos manteniendo la estructura 3D fija.

Referencia: Dauparas et al. (2022), Science.
Repo: https://github.com/dauparas/ProteinMPNN

En el pipeline AI.zymes: 30% del rediseño usa ProteinMPNN (scaffold), 70% RosettaDesign (sitio activo).

## 0. Imports y setup

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')
sys.path.insert(0, '/content/ProteinMPNN')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path

print('Imports OK')

# Verificar que ProteinMPNN esta disponible
PMPNN_DIR = Path('/content/ProteinMPNN')
if PMPNN_DIR.exists():
    print(f'ProteinMPNN disponible en: {PMPNN_DIR}')
else:
    print('ProteinMPNN no encontrado. Ejecutar celda de instalacion.')

## 1. Preparar estructura de entrada

In [ ]:
# Usar estructura predicha por ESMFold o PDB experimental
INPUT_PDB = '/content/esmfold_results/4RUB_predicted.pdb'  # o PDB experimental
OUTPUT_DIR = Path('/content/proteinmpnn_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not os.path.exists(INPUT_PDB):
    # Fallback: usar PDB experimental
    INPUT_PDB = '/content/pdb/chain_a/4RUB_A.pdb'

print(f'Estructura de entrada: {INPUT_PDB}')
print(f'Directorio de salida: {OUTPUT_DIR}')

## 2. Ejecutar ProteinMPNN

In [ ]:
# ProteinMPNN se ejecuta via linea de comandos
# Genera multiples secuencias rediseñadas

PMPNN_SCRIPT = PMPNN_DIR / 'protein_mpnn_run.py'

if not PMPNN_SCRIPT.exists():
    print('Script de ProteinMPNN no encontrado. Clonar repo primero:')
    print('!git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN')
else:
    cmd = [
        'python', str(PMPNN_SCRIPT),
        '--pdb_path', INPUT_PDB,
        '--out_folder', str(OUTPUT_DIR),
        '--num_seq_per_target', '8',
        '--sampling_temp', '0.1',
        '--seed', '42',
        '--batch_size', '1',
    ]
    
    print('Ejecutando ProteinMPNN...')
    print(f'Comando: {" ".join(cmd)}')
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print('Errors:', result.stderr)

## 3. Parsear resultados

In [ ]:
# ProteinMPNN genera archivos FASTA con las secuencias rediseñadas
fasta_files = list(OUTPUT_DIR.glob('*.fa'))

sequences = []
for fasta in fasta_files:
    with open(fasta) as f:
        lines = f.readlines()
    
    for i in range(0, len(lines), 2):
        if i + 1 < len(lines):
            header = lines[i].strip()
            seq = lines[i+1].strip()
            sequences.append({
                'header': header,
                'sequence': seq,
                'length': len(seq),
            })

df_seqs = pd.DataFrame(sequences)
print(f'Secuencias generadas: {len(df_seqs)}')
if not df_seqs.empty:
    display(df_seqs.head())

## 4. Comparar secuencias rediseñadas con la original

In [ ]:
# Secuencia original de 4RUB
ORIGINAL_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

def sequence_identity(seq1, seq2):
    min_len = min(len(seq1), len(seq2))
    matches = sum(1 for a, b in zip(seq1[:min_len], seq2[:min_len]) if a == b)
    return matches / min_len * 100

if not df_seqs.empty:
    df_seqs['identity_pct'] = df_seqs['sequence'].apply(
        lambda s: sequence_identity(s, ORIGINAL_SEQ)
    )
    
    print(f'Identidad con secuencia original:')
    print(f'  Media: {df_seqs["identity_pct"].mean():.1f}%')
    print(f'  Min: {df_seqs["identity_pct"].min():.1f}%')
    print(f'  Max: {df_seqs["identity_pct"].max():.1f}%')
    
    # Distribucion de identidad
    plt.figure(figsize=(8, 4))
    plt.hist(df_seqs['identity_pct'], bins=10, edgecolor='black')
    plt.xlabel('Identidad con secuencia original (%)')
    plt.ylabel('Cantidad de secuencias')
    plt.title('Distribucion de identidad — ProteinMPNN')
    plt.show()

## 5. Guardar mejores secuencias

In [ ]:
if not df_seqs.empty:
    # Guardar todas las secuencias
    output_fasta = OUTPUT_DIR / 'all_sequences.fasta'
    with open(output_fasta, 'w') as f:
        for _, row in df_seqs.iterrows():
            f.write(f'>{row["header"]}\n{row["sequence"]}\n')
    
    print(f'Secuencias guardadas en: {output_fasta}')
    print(f'\nProximo paso:')
    print('- Predecir estructuras de las nuevas secuencias (ESMFold)')
    print('- Evaluar con campos electricos (FieldTools)')
    print('- Seleccion de Boltzmann para elegir las mejores')